In [1]:
!pip install tf_keras

In [2]:
import torch
import tf_keras as keras
import pandas as pd

df = pd.read_csv('movetimes3481887positions.csv')
df.head(10)

2026-02-26 21:15:21.028372: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


,fen,initial_time,increment,time_remaining_for_this_position,time_opponent_took,number_of_legal_moves,in_check,time_taken_for_this_position
0,rnbqkbnr/ppp1pppp/8/3P4/8/8/PPPP1PPP/RNBQKBNR ...,600,0,600.03,3.60,28,False,1.28
1,rnbqkbnr/pp2pppp/2p5/3P4/8/8/PPPP1PPP/RNBQKBNR...,600,0,596.43,1.28,31,False,3.52
2,rnbqkbnr/pp2pppp/2P5/8/8/8/PPPP1PPP/RNBQKBNR b...,600,0,598.75,3.52,32,False,1.44
3,rnbqkbnr/pp3ppp/2P5/4p3/8/8/PPPP1PPP/RNBQKBNR ...,600,0,592.91,1.44,31,False,3.28
4,rnbqkbnr/pP3ppp/8/4p3/8/8/PPPP1PPP/RNBQKBNR b ...,600,0,597.31,3.28,41,False,1.36
5,rn1qkbnr/pb3ppp/8/4p3/8/8/PPPP1PPP/RNBQKBNR w ...,600,0,589.63,1.36,29,False,6.32
6,rn1qkbnr/pb3ppp/8/4p3/8/2N5/PPPP1PPP/R1BQKBNR ...,600,0,595.95,6.32,43,False,1.44
7,rn1qk1nr/pb3ppp/8/2b1p3/8/2N5/PPPP1PPP/R1BQKBN...,600,0,583.31,1.44,32,False,10.16
8,rn1qk1nr/pb3ppp/8/2b1p3/1P6/2N5/P1PP1PPP/R1BQK...,600,0,594.51,10.16,47,False,2.00
9,rn1qk1nr/pb3ppp/1b6/4p3/1P6/2N5/P1PP1PPP/R1BQK...,600,0,573.15,2.00,33,False,13.76


In [3]:
from sklearn.model_selection import train_test_split

nontest, test = train_test_split(df, test_size = 0.1, random_state = 42)

train, val = train_test_split(nontest, test_size = 0.1, random_state = 42)
# given the size of the datatest, cross-validation is unnecessary to maintain a large training dataset.

def drop_fen_column(data_frame):
    return data_frame.drop('fen', axis=1)

# dropping the fen column is done AFTER all the splitting...
# ...because train_test_split is sensitive to the data itself, even if the same random_state is used.
test = drop_fen_column(test)
train = drop_fen_column(train)
val = drop_fen_column(val)

# 'time_taken_for_this_position' is the target feature, so we'll take it out and put it in a variables called y
def X_y_split(dataframe):
    X = dataframe.drop('time_taken_for_this_position', axis=1)
    y = dataframe['time_taken_for_this_position']
    return (X, y)

train_X, train_y = X_y_split(train)
val_X, val_y = X_y_split(val)

import numpy as np

train_X = np.array(train_X, dtype=np.float32)
train_y = np.array(train_y, dtype=np.float32)
val_X   = np.array(val_X,   dtype=np.float32)
val_y   = np.array(val_y,   dtype=np.float32)

In [4]:
from tf_keras import layers

inputs = keras.Input(shape=(6,))
dense_layer = layers.Dense(10, activation='relu')(inputs)
output_layer = layers.Dense(1)(dense_layer)
model = keras.Model(inputs=inputs, outputs=output_layer, name="Single-hidden-layer-NO-FEN")

model.summary()

Model: "Single-hidden-layer-NO-FEN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 6)]               0         
                                                                 
 dense (Dense)               (None, 10)                70        
                                                                 
 dense_1 (Dense)             (None, 1)                 11        
                                                                 
Total params: 81 (324.00 Byte)
Trainable params: 81 (324.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


2026-02-26 21:15:31.712072: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
model.compile(
    loss=keras.losses.MeanSquaredError(),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["mean_squared_error"])

In [6]:
history = model.fit(train_X, train_y, batch_size=64, epochs=10, validation_data=(val_X, val_y))

Epoch 1/10
44068/44068 [==============================] - 64s 1ms/step - loss: 60.4017 - mean_squared_error: 60.4017 - val_loss: 54.0323 - val_mean_squared_error: 54.0323
Epoch 2/10
44068/44068 [==============================] - 52s 1ms/step - loss: 58.0744 - mean_squared_error: 58.0744 - val_loss: 53.9255 - val_mean_squared_error: 53.9255
Epoch 3/10
44068/44068 [==============================] - 52s 1ms/step - loss: 58.0039 - mean_squared_error: 58.0039 - val_loss: 54.8748 - val_mean_squared_error: 54.8748
Epoch 4/10
44068/44068 [==============================] - 52s 1ms/step - loss: 57.7767 - mean_squared_error: 57.7767 - val_loss: 54.7706 - val_mean_squared_error: 54.7706
Epoch 5/10
44068/44068 [==============================] - 52s 1ms/step - loss: 57.7125 - mean_squared_error: 57.7125 - val_loss: 56.3497 - val_mean_squared_error: 56.3497
Epoch 6/10
44068/44068 [==============================] - 55s 1ms/step - loss: 57.7204 - mean_squared_error: 57.7204 - val_loss: 54.0246 - val_me

In [7]:
inputs2 = keras.Input(shape=(6,))
x = layers.Dense(10, activation='relu')(inputs2)
x = layers.Dense(10, activation='relu')(x)
output_layer = layers.Dense(1)(x)
model2 = keras.Model(inputs=inputs2, outputs=output_layer, name="Two-hidden-layers-NO-FEN")

model2.summary()

Model: "Two-hidden-layers-NO-FEN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 6)]               0         
                                                                 
 dense_2 (Dense)             (None, 10)                70        
                                                                 
 dense_3 (Dense)             (None, 10)                110       
                                                                 
 dense_4 (Dense)             (None, 1)                 11        
                                                                 
Total params: 191 (764.00 Byte)
Trainable params: 191 (764.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
model2.compile(
    loss=keras.losses.MeanSquaredError(),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["mean_squared_error"])

In [9]:
history = model2.fit(train_X, train_y, batch_size=64, epochs=10, validation_data=(val_X, val_y))

Epoch 1/10
44068/44068 [==============================] - 54s 1ms/step - loss: 59.4524 - mean_squared_error: 59.4524 - val_loss: 54.3462 - val_mean_squared_error: 54.3462
Epoch 2/10
44068/44068 [==============================] - 54s 1ms/step - loss: 57.6614 - mean_squared_error: 57.6614 - val_loss: 57.1876 - val_mean_squared_error: 57.1876
Epoch 3/10
44068/44068 [==============================] - 54s 1ms/step - loss: 57.4991 - mean_squared_error: 57.4991 - val_loss: 57.9162 - val_mean_squared_error: 57.9162
Epoch 4/10
44068/44068 [==============================] - 54s 1ms/step - loss: 57.3222 - mean_squared_error: 57.3222 - val_loss: 53.6965 - val_mean_squared_error: 53.6965
Epoch 5/10
44068/44068 [==============================] - 54s 1ms/step - loss: 57.3613 - mean_squared_error: 57.3613 - val_loss: 56.1537 - val_mean_squared_error: 56.1537
Epoch 6/10
44068/44068 [==============================] - 54s 1ms/step - loss: 57.4545 - mean_squared_error: 57.4545 - val_loss: 54.2328 - val_me

In [10]:
# Takeaways:
# Even with a bigger neural network, validation loss doesn't improve beyond 53. 
# This hints at a limit with what can be done with the 6 features (without using the fen/board state feature)
# This is better than linear regression (validation loss of 57).

# A random forest regressor with a max depth of 10 gets a validation loss of 53.23. 
# This is further evidence that the we need more features than these 6.

In [11]:
import h5py

In [17]:
model.save('weights', save_format='h5')

model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)

/home/ainan/anaconda3/lib/python3.13/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
# We'll now try a prediction on a single sample from the validation set
val_X[0:1]

array([[300.  ,   0.  , 173.31,   1.36,  42.  ,   0.  ]], dtype=float32)

In [15]:
# The actual time spent on this move
val_y[0]

np.float32(4.48)

In [16]:
# Our model's prediction
model.predict(val_X[0:1])

1/1 [==============================] - 0s 108ms/step


array([[7.546401]], dtype=float32)